# Lab 12: Ingestion at scale

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/sailokeshreddynandavarapu/Documents/Sem_5_Fall_26/Lab_Activity/Lab2/DAMG-7245-Fall-2026 | LLM mode: replay


Set `ONLINE = True` (and put your team name and Northeastern email in `H`) to call EDGAR for real. Offline, the slide code below reads structure samples from `data/edgar_fixture/` (fictional values).

In [2]:
ONLINE = True
import requests
class _FixtureResponse:
    def __init__(self, path): self.path = path
    def json(self): return json.load(open(self.path))
def _offline_get(url, headers=None, timeout=None):
    if "submissions/CIK" in url: return _FixtureResponse("data/edgar_fixture/submissions_SAMPLE.json")
    if "companyfacts/CIK" in url: return _FixtureResponse("data/edgar_fixture/companyfacts_SAMPLE.json")
    raise RuntimeError(f"offline: no fixture for {url}")
if not ONLINE:
    requests.get = _offline_get

In [3]:
# From the slide: The submissions API: find exactly the filings you need
import requests
H = {"User-Agent": "NEU-DAMG7245-2 nandavarapu.s@northeastern.edu"}

def submissions(cik: str) -> dict:
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    return requests.get(url, headers=H, timeout=30).json()

def earnings_8ks(cik: str, n: int = 4) -> list[dict]:
    r = submissions(cik)["filings"]["recent"]  # columns
    rows = zip(r["form"], r["items"], r["filingDate"],
               r["accessionNumber"], r["primaryDocument"])
    out = [dict(form=f, items=i, date=d, accn=a, doc=p)
           for f, i, d, a, p in rows
           if f == "8-K" and "2.02" in i.split(",")]
    return out[:n]         # newest first

In [ ]:
earnings_8ks("0000000001")

In [4]:
# From the slide: companyfacts at scale: facts, not documents
import pandas as pd
KEEP = ("start", "end", "val", "fy", "fp", "form",
        "accn", "frame")

def facts_frame(cik: str) -> pd.DataFrame:
    url = ("https://data.sec.gov/api/xbrl/"
           f"companyfacts/CIK{cik}.json")
    j = requests.get(url, headers=H, timeout=30).json()
    rows = []
    for tax, concepts in j["facts"].items():
        for concept, body in concepts.items():
            for unit, obs in body["units"].items():
                for o in obs:
                    r = {k: o.get(k) for k in KEEP}
                    r.update(cik=cik, taxonomy=tax,
                             concept=concept, unit=unit)
                    rows.append(r)
    return pd.DataFrame(rows)

In [ ]:
facts_frame("0000000001")[["concept", "unit", "end", "val", "fp", "form"]]

The data room arrives as CS1-style records (`data/dataroom/deal_a/export/records.jsonl`), already in `lantern/2.0`. Value-tiered routing decides which path each source takes:

In [5]:
print(llm.P["ingest"]["routing"])
rec = sd.records("deal_a")[0]
print({k: rec[k] for k in ("schema", "doc_id", "deal_id", "source_type", "tier", "page", "block_id", "citation")})

{'comps': 'xbrl+docling_items', 'dataroom': 'cs1_full+docling+ocr', 'web': 'html_text'}
{'schema': 'lantern/2.0', 'doc_id': 'DR-deal_a-001', 'deal_id': 'deal_a', 'source_type': 'dataroom', 'tier': 4, 'page': 1, 'block_id': 'p0001_b001', 'citation': {'kind': 'pdf_bbox', 'doc_id': 'DR-deal_a-001', 'page': 1, 'bbox': [72.0, 91.5, 528.8, 143.0]}}


In [7]:
# From the slide: The synthetic web: crawl from the sitemap, inside an allowlist
import hashlib, xml.etree.ElementTree as ET
from urllib.parse import urlparse
from bs4 import BeautifulSoup
SM = "{http://www.sitemaps.org/schemas/sitemap/0.9}"

def sitemap_urls(xml_text):
    return [l.text.strip()
            for l in ET.fromstring(xml_text).iter(f"{SM}loc")]

def allowed(url, hosts):            # refuse everything else
    u = urlparse(url)
    return u.scheme == "https" and u.hostname in hosts

def web_record(url, html, deal_id):
    soup = BeautifulSoup(html, "html.parser")
    for t in soup(["script", "style", "nav"]):
        t.decompose()
    text = " ".join(soup.get_text(" ").split())
    sha = hashlib.sha256(html.encode()).hexdigest()
    return {"schema": "lantern/2.0", "doc_id": f"WEB-{sha[:8]}",
            "deal_id": deal_id, "source_type": "synthetic_web",
            "tier": 5, "text": text, "source_path": url,
            "citation": {"kind": "web_span", "url": url},
            "lineage": {"parents": [], "source_sha256": sha},
            "ingest_flags": []}

In [ ]:
# Offline, URLs on the course host map to files in data/web_site/
from urllib import robotparser
HOSTS = {llm.P["web"]["host"]}
urls = sitemap_urls(open("data/web_site/sitemap.xml").read())
rp = robotparser.RobotFileParser(); rp.parse(open("data/web_site/robots.txt").read().splitlines())
web = []
for u in urls:
    if not allowed(u, HOSTS) or not rp.can_fetch("LANTERN", u):
        print("refused:", u); continue
    html = open("data/web_site/" + urlparse(u).path.lstrip("/")).read()
    web.append(web_record(u, html, "deal_a"))
print(len(web), "web records;", web[0]["doc_id"], "tier", web[0]["tier"], "|", web[0]["text"][:60])

**Exercise 12.** Point `earnings_8ks` at your CS1 company with `ONLINE = True`. How many 8-K filings report Item 2.02 in the last year?

In [8]:
import time
import pandas as pd

START = "2025-10-14"
END = "2026-10-14"

def earnings_8ks(cik, start=START, end=END):
    data = submissions(cik)
    batches = [data["filings"]["recent"]]

    # Include archived batches that overlap our date window.
    for archive in data["filings"].get("files", []):
        if archive["filingFrom"] <= end and archive["filingTo"] >= start:
            time.sleep(0.2)
            response = requests.get(
                f"https://data.sec.gov/submissions/{archive['name']}",
                headers=H,
                timeout=30,
            )
            response.raise_for_status()
            batches.append(response.json())

    matches = []
    for batch in batches:
        for form, items, date, accession, document in zip(
            batch["form"],
            batch["items"],
            batch["filingDate"],
            batch["accessionNumber"],
            batch["primaryDocument"],
        ):
            item_codes = {item.strip() for item in items.split(",")}
            if form == "8-K" and "2.02" in item_codes and start <= date <= end:
                matches.append({
                    "date": date,
                    "form": form,
                    "items": items,
                    "accession": accession,
                    "document": document,
                })

    return (
        pd.DataFrame(matches, columns=[
            "date", "form", "items", "accession", "document"
        ])
        .drop_duplicates("accession")
        .sort_values("date")
        .reset_index(drop=True)
    )

apple_filings = earnings_8ks("0000320193")
display(apple_filings)

print(
    f"Apple filed {len(apple_filings)} Form 8-K reports containing "
    f"Item 2.02 between {START} and {END}, inclusive."
)

,date,form,items,accession,document
0,2025-10-30,8-K,"2.02,9.01",0000320193-25-000077,aapl-20251030.htm
1,2026-01-29,8-K,"2.02,9.01",0000320193-26-000005,aapl-20260129.htm
2,2026-04-30,8-K,"2.02,9.01",0000320193-26-000011,aapl-20260430.htm
3,2026-07-30,8-K,"2.02,9.01",0000320193-26-000018,aapl-20260730.htm


Apple filed 4 Form 8-K reports containing Item 2.02 between 2025-10-14 and 2026-10-14, inclusive.
